# Pauli-Propagator: Showcase

Two examples below:
1. **A tiny 3-qubit circuit**: propagation cross-checked against PennyLane's own statevector simulator.
2. **VQE on a 64-qubit 2D Ising model**: a size that's out of reach for statevector simulation.

In [ ]:
# Warnings suppressions, purely cosmetic
import warnings
import logging

from pennylane.exceptions import PennyLaneDeprecationWarning
warnings.filterwarnings("ignore", category=PennyLaneDeprecationWarning)

logging.getLogger("jax._src.xla_bridge").setLevel(logging.ERROR)

In [ ]:
import pennylane as qml                  # To define the circuit
from pprop.propagator import Propagator  # For Pauli Propagation (Rust-backed)

### Example 1: A small circuit, checked against PennyLane

A 3-qubit circuit: an RX/RY layer, a `CNOT` entangler, a cosmetic `Barrier`
(drawing only - no physical effect), and a second RX/RY layer. We propagate
`⟨Z₀⟩` backwards through it, then verify the result against PennyLane's own
`default.qubit` statevector simulator.

In [ ]:
# 3-qubit ansatz: RX/RY layer -> CNOT entangler -> Barrier (drawing only) -> RX/RY layer
# Function of parameters, must return a list of qml.expval of observables
def ansatz(params : list[float]):
    qml.RX(params[0], wires=0)
    qml.RX(params[1], wires=1)
    qml.RX(params[2], wires=2)

    qml.RY(params[3], wires=0)
    qml.RY(params[4], wires=1)
    qml.RY(params[5], wires=2)

    qml.CNOT(wires = [2, 1])
    qml.CNOT(wires = [1, 0])
    
    qml.Barrier()

    qml.RX(params[6], wires=0)
    qml.RX(params[7], wires=1)
    qml.RX(params[8], wires=2)

    qml.RY(params[9] , wires=0)
    qml.RY(params[10], wires=1)
    qml.RY(params[11], wires=2)
    
    return [qml.expval(qml.PauliZ(0))] # ⟨Z₀⟩

Constructing a `Propagator`:

In [ ]:
prop = Propagator(ansatz)
print(prop)
prop.show()

Now propagate `Z₀` backwards through the circuit (Heisenberg picture). 
With `k1=k2=None` this is **exact**: no Pauli-weight or frequency truncation.

In [ ]:
prop.propagate() # No weight, frequency cutoffs

The result of propagation is a closed-form symbolic expression for
`⟨Z₀⟩(θ)` - a sum of products of `sin`/`cos` of the circuit parameters.
`prop.expression()` renders it as a SymPy expression

In [ ]:
prop.expression()  # defaults to idx=0, i.e. ⟨Z₀⟩

Evaluating the propagated expression at a parameter point is just
plugging numbers into that closed form.

In [ ]:
random_params = qml.numpy.arange(prop.num_params)
prop(random_params)

`eval_and_grad` returns the value *and* the analytic gradient in one
call: the gradient falls out of the same closed-form expression
(derivatives of `sin`/`cos`), no parameter-shift rule or autodiff needed.
This is what `pprop.optimization.adam` uses internally.

In [ ]:
prop.eval_and_grad(random_params)

**Sanity check:** build the same circuit as a plain PennyLane `QNode` on
a statevector device and compare `⟨Z₀⟩` to the value pprop computed above -
they should match to numerical precision.

In [ ]:
# Should match prop(random_params) above
device = qml.device('default.qubit', wires = 3)

circuit = qml.QNode(ansatz, device)
pennylane_output = circuit(random_params)
pennylane_output

### Example 2: VQE on a 64-qubit 2D Ising model

A more realistic use case: variational ground-state search for the 2D
transverse-field Ising model (TFIM) on an 8x8 open-boundary lattice (64
qubits), far beyond what a statevector simulator can hold (`2^64`
amplitudes). `pprop` makes this tractable by propagating only the
Hamiltonian's Pauli terms backwards through the ansatz.

The Hamiltonian in full (nearest-neighbour `ZZ` coupling `J` and transverse field `h`,
both normalised by the number of qubits `N`):

$$H(J, h) = -\frac{1}{N}\left(J\sum_{\langle i,j \rangle} Z_i Z_j + h\sum_i X_i\right)$$

<img src="./assets/ising2d.svg" width="600">

*The 8x8 lattice: qubits sit on grid sites, `ZZ` couplings link nearest
neighbours (open boundaries), and every qubit also feels the transverse
field `h`.*

In [ ]:
# adam: pprop's Adam-based optimiser, drives Propagator.eval_and_grad directly
from pprop.optimization import adam
import matplotlib.pyplot as plt

In [ ]:
SIDE = 8
J = 1
H = 1

NUM_QUBITS : int = SIDE * SIDE

# Builds the TFIM Hamiltonian above as a qml.Hamiltonian (open boundaries:
# only the y < SIDE-1 / x < SIDE-1 neighbour pairs are coupled).
def hamiltonian() -> qml.Hamiltonian:
    coeffs = []
    obs = []

    # Nearest-neighbor ZZ interactions
    for x in range(SIDE):
        for y in range(SIDE):
            i = x * SIDE + y

            # Right neighbor
            if y < SIDE - 1:
                j = x * SIDE + (y + 1)
                coeffs.append(-J / (SIDE*SIDE))
                obs.append(qml.PauliZ(i) @ qml.PauliZ(j))

            # Down neighbor
            if x < SIDE - 1:
                j = (x + 1) * SIDE + y
                coeffs.append(-J / (SIDE*SIDE))
                obs.append(qml.PauliZ(i) @ qml.PauliZ(j))

    # Transverse-field X terms
    for i in range(SIDE*SIDE):
        coeffs.append(-H / (SIDE*SIDE))
        obs.append(qml.PauliX(i))

    return qml.Hamiltonian(coeffs, obs)

# Hardware-efficient ansatz: RY+RX on every qubit, brick-wall CNOTs
# (horizontal then vertical, 2 layers each), RX layer, RY layer, then the
# Hamiltonian expectation value.
def circuit_vqe(params):
    index = 0

    # Initial RY and RX
    for q in range(SIDE*SIDE):
        qml.RY(params[index], wires=q)
        index += 1
        qml.RX(params[index], wires=q)
        index += 1

    # Horizontal entanglers
    for d in range(2):
        y_start = 0 if d % 2 == 0 else 1
        for x in range(SIDE):
            for y in range(y_start, SIDE - 1, 2):
                i = x * SIDE + y
                j = x * SIDE + (y + 1)
                qml.CNOT(wires=[i, j])

    # RX layer
    for q in range(SIDE*SIDE):
        qml.RX(params[index], wires=q)
        index += 1

    # Vertical entanglers
    for d in range(2):
        x_start = 0 if d % 2 == 0 else 1
        for y in range(SIDE):
            for x in range(x_start, SIDE - 1, 2):
                i = x * SIDE + y
                j = (x + 1) * SIDE + y
                qml.CNOT(wires=[i, j])

    # RX layer
    for q in range(SIDE*SIDE):
        qml.RX(params[index], wires=q)
        index += 1

    # Final RY
    for q in range(SIDE*SIDE):
        qml.RY(params[index], wires=q)
        index += 1
        
    return qml.expval(hamiltonian())

Same `Propagator` API as Example 1, just at 64 qubits - construction is
still just recording the tape, so this stays fast regardless of qubit
count.

In [ ]:
prop_vqe = Propagator(circuit_vqe)
print(prop_vqe)
prop_vqe.show()

In [ ]:
# Exact propagation + exact structural pruning
prop_vqe.propagate(use_dead_qubit_pruner=True, use_xy_weight_pruner=True)

Optimise the ansatz parameters to minimise the Hamiltonian's expectation
value. `adam` calls `prop_vqe.eval_and_grad` directly - value and analytic
gradient from the same closed-form expression, no parameter-shift rule and
no statevector simulation involved.

In [ ]:
# L(f) = f[0] because circuit_vqe returns a single observable: the Hamiltonian itself
result = adam(
    L=lambda f: f[0],
    propagator=prop_vqe,
    params_init=qml.numpy.random.rand(prop_vqe.num_params),
    lr=1e-2,
    num_steps=300
)

Compare the optimised energy to a reference DMRG (density-matrix
renormalisation group) ground-state energy for this lattice size - DMRG is
a strong classical baseline for 2D TFIM ground states.

In [ ]:
DMRG_ENERGY = -1.8996301853818673 # Obtained through DMRG: chi: 1024, sweeps: 15
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(result['loss_history'])
ax.axhline(DMRG_ENERGY, color='r', linestyle='--', label=f'DMRG ($E = {DMRG_ENERGY:.6f}$)')
ax.set_xlabel('Step')
ax.set_ylabel('Energy')
ax.set_title(f'VQE convergence 2D TFIM ${SIDE} \\times {SIDE}$')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Final energy : {result['fun']:.6f}")
print(f"DMRG energy  : {DMRG_ENERGY:.6f}")
print(f"Gap          : {result['fun'] - DMRG_ENERGY:.6f}")